# 📅 Day 16 — Mock Interview Practice

**Week 3 · Day 2 · Level: ADVANCED (deepening into HARD)**

Week 3 continues: **product-company level** questions — complex end-to-end scenarios, optimization trade-offs, and system-design aspects. 18 questions across 8 categories — 14 [ADVANCED] + 4 [HARD]. Today pushes deeper into gaps-and-islands sessionization, skewed data handling, exactly-once semantics, SCD Type 6, and circuit breakers. Every answer should come with a trade-off articulated out loud.

| # | Category | Questions | Level |
| --- | --- | --- | --- |
| 1 | SQL Advanced | Q1-Q2 | ADVANCED / HARD |
| 2 | Python — Loop/Function/List/Dict | Q3-Q4 | ADVANCED |
| 3 | PySpark Coding | Q5-Q6 | ADVANCED |
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | ADVANCED |
| 5 | ETL — Failures, Late Data, Schema Handling | Q9-Q11 | ADVANCED / HARD |
| 6 | Data Modeling | Q12-Q13 | ADVANCED |
| 7 | Snowflake | Q14-Q15 | ADVANCED |
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | ADVANCED / HARD |

**Difficulty progression**: Day 15 introduced Week 3 with z-score bands, range joins, caching layers, and bitemporal facts. Day 16 deepens with cohort retention grids, sessionization via gaps-and-islands, skewed-data salting, SCD Type 6 hybrids, exactly-once end-to-end, and circuit breakers — each a step more complex than its Day 15 counterpart.

## Q1 [ADVANCED] SQL Advanced — Cohort Retention Grid from Activity Events

**Scenario**: A product analytics team needs a **cohort retention grid**: group users by their signup month (the cohort), then for each subsequent month compute what % of the cohort was active. This is the foundational query behind every retention dashboard — and interviewers love it because it requires multi-level aggregation with conditional logic.

**Requirements**:
1. Build the cohort retention grid: rows = cohort month, columns = months since signup (0, 1, 2, …), values = % of cohort active in that month (cohort_month_size = users active in month 0).
2. Month 0 retention is always 100% (every user is active in their signup month by definition).
3. Handle cohorts of different sizes — a 3-user cohort with 2 active in month 1 = 66.7%, not the same as a 100-user cohort with 2 active.
4. Show the grid as a pivot with explicit month-since-signup columns.

**Sample data**: 7 users across 3 signup cohorts (Jan, Feb, Mar 2026), with activity spanning 4 months. Expected: Jan cohort (3 users) shows 100% / 67% / 33% / 33% for months 0-3.

In [0]:
# ── Solution Q1: cohort retention grid via conditional aggregation + pivot ─────────
from pyspark.sql.functions import col, min as spark_min, countDistinct, when, lit
from pyspark.sql.types import *

# Sample: (user_id, signup_month, active_month)
activity = [
    (1, "2026-01", "2026-01"), (1, "2026-01", "2026-02"), (1, "2026-01", "2026-04"),
    (2, "2026-01", "2026-01"), (2, "2026-01", "2026-02"),
    (3, "2026-01", "2026-01"), (3, "2026-01", "2026-03"),
    (4, "2026-02", "2026-02"), (4, "2026-02", "2026-03"),
    (5, "2026-02", "2026-02"),
    (6, "2026-03", "2026-03"), (6, "2026-03", "2026-04"),
    (7, "2026-03", "2026-03"),
]

raw = spark.createDataFrame(activity, ["user_id", "signup_month", "active_month"])

# Step 1: cohort size = distinct users per signup month (month-0 denominator)
cohort_sizes = (raw.groupBy("signup_month")
                  .agg(countDistinct("user_id").alias("cohort_size")))

# Step 2: months_since_signup = months between active and signup
with_gap = (raw.withColumn("months_since",
             (col("active_month").substr(1, 4).cast("int") * 12 + col("active_month").substr(6, 2).cast("int"))
             - (col("signup_month").substr(1, 4).cast("int") * 12 + col("signup_month").substr(6, 2).cast("int"))
            ))

# Step 3: distinct active users per (cohort, months_since)
retention = (with_gap.groupBy("signup_month", "months_since")
               .agg(countDistinct("user_id").alias("active_users")))

# Step 4: join cohort sizes and compute retention rate
retention_pct = (retention.join(cohort_sizes, "signup_month")
                   .withColumn("retention_pct", (col("active_users") / col("cohort_size") * 100).cast("decimal(5,1)")))

# Step 5: pivot into grid
grid = (retention_pct.groupBy("signup_month")
         .pivot("months_since", [0, 1, 2, 3])
         .agg(spark_min("retention_pct"))
         .orderBy("signup_month"))

grid.show()

# Verify: Jan cohort (3 users) month 0 = 100%, month 1 = 2/3 = 66.7%, month 2 = 1/3 = 33.3%
jan = grid.filter("signup_month = '2026-01'").first()
assert jan["0"] == 100.0, f"Month 0 should be 100%, got {jan['0']}"
assert jan["1"] == 66.7, f"Month 1 should be 66.7%, got {jan['1']}"
assert jan["2"] == 33.3, f"Month 2 should be 33.3%, got {jan['2']}"
print("Cohort retention grid verified ✓")
print("\nInterview insight: the key is the DENOMINATOR — cohort_size is fixed at month 0.")
print("A common mistake is dividing by current-month active count instead of original cohort size.")

## Q2 [HARD] SQL Advanced — Sessionization via Gaps-and-Islands

**Scenario**: You have a clickstream of user events with timestamps. A **session** is defined as a sequence of events by the same user where no two consecutive events are more than 30 minutes apart. If a user clicks at 10:00, 10:20, 11:00, that's two sessions: [10:00, 10:20] and [11:00] (40-min gap between 10:20 and 11:00). This is a classic gaps-and-islands problem asked at Meta, Google, and Amazon.

**Requirements**:
1. For each user, identify session boundaries using the 30-minute inactivity gap rule.
2. Assign a `session_id` to each event — events in the same session share the same id.
3. Produce a per-session summary: user_id, session_start, session_end, event_count, session_duration_minutes.
4. Handle edge cases: a single event is a session (duration 0), events exactly 30 minutes apart are in the SAME session (≤ 30 min = same session).

**Sample data**: user 1 has events at 10:00, 10:20, 11:00 (gap = 40 min → new session), 11:05, 12:00 (gap = 55 min → new session). Expected: 3 sessions for user 1.

In [0]:
# ── Solution Q2: sessionization via gaps-and-islands with window functions ──────
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, when, sum as spark_sum, min as spark_min, max as spark_max, count, unix_timestamp, expr

# Sample clickstream: (event_id, user_id, event_ts)
events = [
    (1, 1, "2026-01-01 10:00:00"),
    (2, 1, "2026-01-01 10:20:00"),
    (3, 1, "2026-01-01 11:00:00"),   # 40-min gap → new session
    (4, 1, "2026-01-01 11:05:00"),
    (5, 1, "2026-01-01 12:00:00"),   # 55-min gap → new session
    (6, 2, "2026-01-01 09:00:00"),
    (7, 2, "2026-01-01 09:25:00"),   # 25-min gap → same session
    (8, 2, "2026-01-01 10:00:00"),   # 35-min gap → new session
]

df = spark.createDataFrame(events, ["event_id", "user_id", "event_ts"])
df = df.withColumn("event_ts", col("event_ts").cast("timestamp"))

# Step 1: lag to get previous event timestamp per user
w = Window.partitionBy("user_id").orderBy("event_ts")
df = df.withColumn("prev_ts", lag("event_ts").over(w))

# Step 2: gap in minutes; flag new session when gap > 30 or no previous (first event)
df = (df.withColumn("gap_minutes",
        when(col("prev_ts").isNotNull(),
             (unix_timestamp("event_ts") - unix_timestamp("prev_ts")) / 60)
        .otherwise(0))
      .withColumn("is_new_session", when(col("gap_minutes") > 30, 1).otherwise(0)))

# Step 3: cumulative sum of is_new_session → session number within user
w2 = Window.partitionBy("user_id").orderBy("event_ts").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
df = df.withColumn("session_num", spark_sum("is_new_session").over(w2))

# Step 4: global session_id (user_id + session_num for uniqueness)
df = df.withColumn("session_id", expr("concat(user_id, '-', session_num + 1)"))

# Step 5: per-session summary
session_summary = (df.groupBy("user_id", "session_id")
                     .agg(spark_min("event_ts").alias("session_start"),
                          spark_max("event_ts").alias("session_end"),
                          count("event_id").alias("event_count"))
                     .withColumn("duration_minutes",
                        (unix_timestamp("session_end") - unix_timestamp("session_start")) / 60)
                     .orderBy("user_id", "session_start"))

session_summary.show(truncate=False)

# Verify: user 1 has 3 sessions, user 2 has 2 sessions
assert session_summary.filter("user_id = 1").count() == 3, "User 1 should have 3 sessions"
assert session_summary.filter("user_id = 2").count() == 2, "User 2 should have 2 sessions"
print("Sessionization verified ✓ — 3 sessions for user 1, 2 sessions for user 2")
print("\nGaps-and-islands pattern: lag → gap → flag → cumulative sum → group.")
print("The cumulative sum of 'new session' flags creates a monotonically increasing session number.")

## Q3 [ADVANCED] Python — LRU Cache Implementation (Doubly-Linked List + Dict)

**Scenario**: You're building a caching layer for a recommendation service that fetches user profiles from an external API. The cache must evict the **least recently used** entry when capacity is exceeded. Implement it from scratch — no `functools.lru_cache` allowed.

**Requirements**:
1. Implement `LRUCache(capacity)` with `get(key)` and `put(key, value)` — both O(1).
2. Use a **doubly-linked list** for O(1) recency updates + a **dict** for O(1) key lookup.
3. `get` moves the accessed node to the head (most recently used). Returns -1 if key not found.
4. `put` inserts/updates at head. If capacity exceeded, evicts the tail (LRU).
5. Include test cases: basic ops, eviction order, capacity-1 edge case, update existing key.

**Expected**: cache(2); put(1,'a'), put(2,'b'), get(1), put(3,'c'), get(2) → -1 (2 was LRU after 1 was accessed).

In [0]:
# ── Solution Q3: LRU cache with doubly-linked list + dict ────────────────────

class DLinkedNode:
    __slots__ = ('key', 'value', 'prev', 'next')  # lean nodes for memory efficiency
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> DLinkedNode
        # Sentinel head/tail to avoid null checks on insert/remove
        self.head = DLinkedNode()
        self.tail = DLinkedNode()
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node):
        """Unlink a node from the list."""
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_head(self, node):
        """Insert right after the head sentinel (most recently used)."""
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _move_to_head(self, node):
        self._remove(node)
        self._add_to_head(node)

    def _pop_tail(self):
        """Evict and return the LRU node (right before tail sentinel)."""
        lru = self.tail.prev
        self._remove(lru)
        return lru

    def get(self, key):
        if key not in self.cache:
            return -1
        self._move_to_head(self.cache[key])  # mark as most recently used
        return self.cache[key].value

    def put(self, key, value):
        if key in self.cache:
            self.cache[key].value = value
            self._move_to_head(self.cache[key])
        else:
            node = DLinkedNode(key, value)
            self.cache[key] = node
            self._add_to_head(node)
            if len(self.cache) > self.capacity:
                evicted = self._pop_tail()
                del self.cache[evicted.key]

# ── Test cases ────────────────────────────────────────────────────────────────
c = LRUCache(2)
c.put(1, 'a')
c.put(2, 'b')
assert c.get(1) == 'a'         # access 1 → 1 is now MRU, 2 is LRU
c.put(3, 'c')                  # evicts 2 (the LRU)
assert c.get(2) == -1          # 2 was evicted
assert c.get(3) == 'c'
assert c.get(1) == 'a'         # 1 still there
c.put(4, 'd')                  # evicts 3 (now LRU after 1 was accessed)
assert c.get(3) == -1
assert c.get(4) == 'd'
assert c.get(1) == 'a'
print("All LRU cache tests passed ✓")

# Edge case: capacity = 1
c1 = LRUCache(1)
c1.put(1, 'x')
c1.put(2, 'y')
assert c1.get(1) == -1
assert c1.get(2) == 'y'
print("Capacity-1 edge case passed ✓")

# Edge case: update existing key doesn't evict
c2 = LRUCache(2)
c2.put(1, 'a')
c2.put(2, 'b')
c2.put(1, 'updated')           # update, not insert → no eviction
assert c2.get(1) == 'updated'
assert c2.get(2) == 'b'
print("Update-existing-key edge case passed ✓")

## Q4 [ADVANCED] Python — Sliding Window Maximum (Monotonic Deque)

**Scenario**: A real-time monitoring system needs the **maximum value in every sliding window of size k** over a stream of sensor readings. For a window [3, 1, 5] with k=3, the answer is 5. The naive approach (scan each window) is O(n·k); the interviewer expects O(n).

**Requirements**:
1. Implement `max_sliding_window(nums, k)` that returns a list of the max in each window position.
2. Use a **monotonic deque** (decreasing) storing indices — front is always the current max.
3. Explain why the deque stays O(n) despite the inner while loop (amortized: each element enters and leaves the deque at most once).
4. Handle edge cases: k=1 (return nums), k=len(nums) (return [max(nums)]), empty input.

**Expected**: `max_sliding_window([1,3,-1,-3,5,3,6,7], 3)` → `[3,3,5,5,6,7]`.

In [0]:
# ── Solution Q4: sliding window maximum with monotonic deque ────────────────
from collections import deque

def max_sliding_window(nums, k):
    """
    Monotonic decreasing deque of INDICES.
    - Front of deque = index of the max in the current window.
    - Back of deque = smallest element still relevant.
    - We pop from back any element ≤ new element (they can never be the max
      while the new element is in the window).
    - We pop from front if the index is outside the window.
    Amortized O(n): each index enters and leaves the deque at most once.
    """
    if not nums or k == 0:
        return []
    if k == 1:
        return list(nums)

    dq = deque()          # stores indices, values are monotonically decreasing
    result = []

    for i, val in enumerate(nums):
        # 1. Remove indices that are out of the current window (left side)
        if dq and dq[0] < i - k + 1:
            dq.popleft()

        # 2. Remove from back all elements ≤ current value (they're dominated)
        while dq and nums[dq[-1]] <= val:
            dq.pop()

        # 3. Add current index
        dq.append(i)

        # 4. Once we have a full window, record the max (front of deque)
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result

# ── Test cases ────────────────────────────────────────────────────────────────
assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window([1], 1) == [1]
assert max_sliding_window([5, 4, 3, 2, 1], 3) == [5, 4, 3]   # decreasing
assert max_sliding_window([1, 2, 3, 4, 5], 3) == [3, 4, 5]   # increasing
assert max_sliding_window([1, 3, 1, 2, 0, 5], 3) == [3, 3, 2, 2, 5]
assert max_sliding_window([], 3) == []
assert max_sliding_window([1, 2], 2) == [2]                   # k = len(nums)
print("All sliding window maximum tests passed ✓")

print("\nWhy O(n) despite the inner while loop:")
print("  Each element is pushed onto the deque exactly once and popped at most once.")
print("  Total pop operations across the entire algorithm ≤ n, so amortized O(1) per element.")
print("\nKey insight: the deque stores indices, not values, so we can check window boundaries.")
print("A decreasing deque means the front is ALWAYS the max — no need to re-scan.")

## Q5 [ADVANCED] PySpark Coding — Handling Skewed Data with Salting

**Scenario**: You have a `facts` table where 80% of rows have `category = 'ELECTRONICS'` and 20% are spread across 99 other categories. A join with `dim_categories` on `category` causes one task to process the entire ELECTRONICS partition while the other 99 tasks finish in seconds. The classic **data skew** problem.

**Requirements**:
1. Implement the **salting technique**: append a random salt suffix (0..N-1) to the skewed key on the large table, and explode the small table N times with matching salts.
2. After the join, strip the salt to recover original rows.
3. Show the before (skewed) and after (salted) partition distribution.
4. Explain when salting wins vs broadcast join vs AQE skew splitting (Spark 3+/serverless).

**Sample data**: 100 facts rows (85 with 'ELECTRONICS', 15 spread across others). dim has 100 rows (1 per category). Salt factor = 5.

In [0]:
# ── Solution Q5: salting technique for skewed joins ────────────────────────
from pyspark.sql.functions import col, concat, lit, rand, floor, explode, array, broadcast
from pyspark.sql.types import *
import random

# Build skewed facts: 85 'ELECTRONICS', 5 each for 'BOOKS', 'CLOTHING', 'FOOD', 'TOYS'
fact_rows = [(i, "ELECTRONICS") for i in range(85)]
fact_rows += [(i, c) for c in ["BOOKS", "CLOTHING", "FOOD", "TOYS"] for i in range(85, 100)]
facts = spark.createDataFrame(fact_rows, ["id", "category"])

dim_rows = [(c, c.lower() + "_dept") for c in
            ["ELECTRONICS", "BOOKS", "CLOTHING", "FOOD", "TOYS"]]
dim = spark.createDataFrame(dim_rows, ["category", "dept_name"])

# ── BEFORE: naive join → all ELECTRONICS rows land in ONE partition ──
naive = facts.join(dim, "category")
print(f"Naive join row count: {naive.count()}")

# ── AFTER: salting technique ──
SALT_N = 5

# 1. Add salt to the LARGE table (facts): random 0..SALT_N-1 appended to key
facts_salted = (facts.withColumn("salt", floor(rand() * SALT_N).cast("int"))
                     .withColumn("salted_key", concat(col("category"), lit("_"), col("salt"))))

# 2. Explode the SMALL table (dim) SALT_N times: one row per salt value
dim_exploded = (dim.withColumn("salt_id", explode(array([lit(i) for i in range(SALT_N)])))
                  .withColumn("salted_key", concat(col("category"), lit("_"), col("salt_id"))))

# 3. Join on the salted key — the 85 ELECTRONICS rows are now spread across 5 partitions
salted_join = facts_salted.join(dim_exploded, "salted_key")

# 4. Strip salt and verify correctness
result = salted_join.select(col("id"), col("category"), col("dept_name")).dropDuplicates(["id", "category"])

result_count = result.count()
assert result_count == 100, f"Expected 100 rows after salting, got {result_count}"
assert result.filter("category = 'ELECTRONICS'").count() == 85
print(f"Salted join row count: {result_count} (100 ✓)")
print(f"ELECTRONICS rows: {result.filter(\"category = 'ELECTRONICS'\").count()} (85 ✓)")

print("\nTalk track — when to use which technique:")
print("  SALTING: manual, works on all Spark versions, best when you KNOW the skewed keys.")
print("  BROADCAST JOIN: if the dim is small enough, broadcast it → no shuffle at all.")
print("  AQE SKEW SPLITTING (Spark 3+/serverless): spark.sql.adaptive.skewJoin.enabled=true")
print("    automatically splits fat partitions — the modern default, but you still need to")
print("    understand salting for interviews and legacy clusters.")
print("  Rule of thumb: try AQE first → if still skewed → broadcast if dim fits → salt as last resort.")

## Q6 [ADVANCED] PySpark Coding — SCD Type 2 MERGE INTO with Delta Lake

**Scenario**: A `dim_customer` table needs to track historical changes to customer attributes. When a customer's `tier` changes from 'STANDARD' to 'GOLD', the old row is expired (`is_current = false`, `end_date = today`) and a new row is inserted with `is_current = true`. This is **SCD Type 2** — essential for point-in-time analysis ("what tier was this customer in last quarter?").

**Requirements**:
1. Create the SCD2 Delta table with: `customer_sk`, `customer_id`, `tier`, `email`, `start_date`, `end_date`, `is_current`.
2. Implement the MERGE INTO logic: on match (same tier) → update; on match (tier changed) → expire old + insert new; on no match → insert new.
3. Show that after two updates, a customer has 3 rows: original (expired), first change (expired), current (active).
4. Demonstrate point-in-time query: "what tier was customer 1 on 2026-01-15?"

**Sample data**: Customer 1 starts as STANDARD on 2026-01-01, changes to GOLD on 2026-01-15, changes to VIP on 2026-02-01.

In [0]:
# ── Solution Q6: SCD Type 2 with Delta MERGE INTO ──────────────────────────────
from pyspark.sql.functions import col, lit, current_date, to_date, when

# Initialize the SCD2 table with the initial load
TARGET = "workspace.default.d16_q6_dim_customer"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

initial = spark.createDataFrame([
    (1, "CUST-001", "STANDARD", "cust1@email.com"),
    (2, "CUST-002", "STANDARD", "cust2@email.com"),
], ["customer_sk", "customer_id", "tier", "email"])

initial = (initial.withColumn("start_date", to_date(lit("2026-01-01")))
                .withColumn("end_date", to_date(lit("9999-12-31")))
                .withColumn("is_current", lit(True)))

initial.write.format("delta").saveAsTable(TARGET)

# ── Function: apply SCD2 update from a batch of source changes ──
def apply_scd2(source_df, target_table, update_date_str):
    """MERGE source changes into the SCD2 target: expire old, insert new."""
    source_df = source_df.withColumn("new_start_date", to_date(lit(update_date_str)))
    source_df.createOrReplaceTempView("scd2_source")

    spark.sql(f"""
    -- Step 1: Expire current rows whose tier changed
    MERGE INTO {target_table} AS t
    USING scd2_source AS s
      ON t.customer_id = s.customer_id AND t.is_current = true
    WHEN MATCHED AND t.tier != s.tier
      THEN UPDATE SET
        t.end_date = s.new_start_date,
        t.is_current = false
    """)

    # Step 2: Insert new current rows for changed attributes
    new_rows = spark.sql(f"""
    SELECT s.customer_id, s.tier, s.email,
           s.new_start_date AS start_date,
           to_date('9999-12-31') AS end_date,
           true AS is_current
    FROM scd2_source s
    LEFT JOIN {target_table} t
      ON t.customer_id = s.customer_id AND t.is_current = true
    WHERE t.customer_id IS NULL
    """)
    new_rows.write.format("delta").mode("append").saveAsTable(target_table)

# ── Update 1: customer 1 → GOLD on 2026-01-15 ──
batch1 = spark.createDataFrame([(1, "CUST-001", "GOLD", "cust1@email.com")],
                               ["customer_id", "customer_id_str", "tier", "email"])
# fix: use proper column names
batch1 = spark.createDataFrame([("CUST-001", "GOLD", "cust1@email.com")],
                               ["customer_id", "tier", "email"])
apply_scd2(batch1, TARGET, "2026-01-15")

# ── Update 2: customer 1 → VIP on 2026-02-01 ──
batch2 = spark.createDataFrame([("CUST-001", "VIP", "cust1@email.com")],
                               ["customer_id", "tier", "email"])
apply_scd2(batch2, TARGET, "2026-02-01")

# ── Verify: customer 1 should have 3 rows, 1 current ──
hist = spark.sql(f"SELECT customer_id, tier, start_date, end_date, is_current FROM {TARGET} WHERE customer_id = 'CUST-001' ORDER BY start_date")
hist.show(truncate=False)

row_count = hist.count()
current_count = hist.filter("is_current = true").count()
assert row_count == 3, f"Expected 3 rows, got {row_count}"
assert current_count == 1, f"Expected 1 current row, got {current_count}"
assert hist.filter("is_current = true").first()["tier"] == "VIP"
print("SCD Type 2 verified ✓ — 3 rows (STANDARD→GOLD→VIP), 1 current")

# ── Point-in-time query: what tier was customer 1 on 2026-01-15? ──
pit = spark.sql(f"""
SELECT tier FROM {TARGET}
WHERE customer_id = 'CUST-001'
  AND start_date <= '2026-01-15' AND end_date > '2026-01-15'
""").collect()[0]["tier"]
assert pit == "GOLD", f"On 2026-01-15, tier should be GOLD, got {pit}"
print(f"Point-in-time query: customer 1 on 2026-01-15 → tier = {pit} ✓")

## Q7 [ADVANCED] Spark Optimization — Adaptive Query Execution (AQE) Internals

**Scenario**: You inherited a pipeline where someone hard-coded `spark.sql.shuffle.partitions = 200`. The team manually tunes partitions for every join. On Spark 3.0+, AQE was enabled and things "just worked better" — but nobody can explain WHY. Your interviewer asks: "Walk me through the three AQE optimizations and when each fires."

**Requirements**:
1. Explain the **three AQE optimizations** in detail: (a) dynamic partition coalescing, (b) dynamic join strategy switch, (c) dynamic skew splitting.
2. For each: what problem it solves, what triggers it, and what config controls it.
3. Explain the **runtime statistics** that AQE collects between stages and why it can only optimize after the first stage completes (not mid-stage).
4. Describe what happens on Databricks serverless (auto-managed) vs classic shared clusters.

**Expected output**: a structured explanation with config knobs and the talk-track for an interview.

In [0]:
# ── Solution Q7: AQE internals — the three optimizations ──────────────────────
print("AQE (Adaptive Query Execution) — Three Optimizations")
print("=" * 70)
print()
print("1. DYNAMIC PARTITION COALESCING")
print("   Problem: spark.sql.shuffle.partitions defaults to 200. A 200MB post-shuffle")
print("   dataset → 200 partitions of ~1MB each → tiny tasks, scheduling overhead dominates.")
print("   Fix: AQE coalesces small adjacent partitions into bigger ones based on actual")
print("   post-shuffle data size. Target: spark.sql.adaptive.advisoryPartitionSizeInBytes")
print("   (default 64MB). Coalesces 200 tiny → ~3-4 well-sized partitions.")
print("   Config: spark.sql.adaptive.coalescePartitions.enabled (default true)")
print("           spark.sql.adaptive.advisoryPartitionSizeInBytes = 64MB")
print()
print("2. DYNAMIC JOIN STRATEGY SWITCH")
print("   Problem: At plan time, Spark estimates a table at 10GB → chooses sort-merge join.")
print("   After the map stage runs, actual size = 50MB → broadcast would have been cheaper.")
print("   Fix: AQE re-evaluates join strategy using ACTUAL statistics from completed stages.")
print("   If a side is now under spark.sql.adaptive.autoBroadcastJoinThreshold (default 10MB,"),
print("   or 0 = use the static threshold), it switches to broadcast.")
print("   Config: spark.sql.adaptive.localShuffleReader.enabled (enables the broadcast switch)")
print()
print("3. DYNAMIC SKEW SPLITTING")
print("   Problem: One partition is 10x larger than others → the straggler task dominates runtime.")
print("   Fix: AQE detects skewed partitions (size > median × spark.sql.adaptive.skewJoin.factor,")
print("   default 5, AND above spark.sql.adaptive.skewJoin.skewedPartitionFactor threshold) and")
print("   splits them into multiple smaller partitions that are processed in parallel.")
print("   Config: spark.sql.adaptive.skewJoin.enabled = true")
print("           spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5")
print()
print("WHY AQE FIRES BETWEEN STAGES, NOT MID-STAGE:")
print("   AQE collects statistics (partition sizes, row counts) from the COMPLETED map stage.")
print("   It can only re-optimize the NEXT stage's plan, not the one already running.")
print("   This is why AQE helps for pipelines with multiple stages — the first stage still runs")
print("   with the static plan, but subsequent stages benefit from runtime stats.")
print()
print("DATABRICKS SERVERLESS vs CLASSIC:")
print("   Serverless: AQE is ON by default with 'auto' partitioning — you don't set")
print("   spark.sql.shuffle.partitions at all; the platform manages it entirely.")
print("   Classic shared: set spark.sql.adaptive.enabled = true explicitly (default true in 3.2+).")

# Show current AQE settings
print("\n── Current AQE settings on this cluster ──")
for c in [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
]:
    try:
        print(f"  {c} = {spark.conf.get(c)}")
    except Exception:
        print(f"  {c} = (not set / not available)")
print("\nInterview one-liner: 'AQE turns Spark from a compile-time optimizer into a")
print("runtime optimizer — it re-plans between stages using actual statistics.'")

## Q8 [ADVANCED] Spark Optimization — Garbage Collection Tuning for Spark Executors

**Scenario**: A Spark job processing 500GB of data is slow. The Spark UI shows long GC pauses (15+ seconds) on executor tasks. The team asks: "Should we switch to G1GC? What about off-heap?"

**Requirements**:
1. Explain why Spark executors GC heavily: the **unified memory pool** (execution + storage), shuffle data on heap, and task concurrency.
2. Compare **ParallelGC** (default for ≤ 4 cores) vs **G1GC** (default for 5+ cores) vs **ZGC** (experimental in Spark 3.3+).
3. Explain **off-heap memory** (`spark.memory.offHeap.enabled`, `spark.memory.offHeap.size`) — what it stores and why it reduces GC pressure.
4. Provide a diagnostic walk-through: what to look for in the Spark UI (GC time ratio, task serialization time) and what config changes to try.

**Expected output**: config recommendations + a triage checklist.

In [0]:
# ── Solution Q8: GC tuning talk-track and config ──────────────────────────────
print("WHY SPARK EXECUTORS GC HEAVILY:")
print("  Spark's unified memory pool (spark.memory.fraction = 0.6) holds both EXECUTION")
print("  memory (shuffle buffers, join sorts, agg hashes) and STORAGE memory (cached blocks).")
print("  When the pool fills, objects are GC'd or spilled. With multiple tasks per executor")
print("  (spark.executor.cores), all tasks share ONE JVM heap — a single memory-hungry task")
print("  can trigger full-GC pauses for ALL tasks on that executor.")
print()
print("GARBAGE COLLECTOR COMPARISON:")
print("─" * 70)
print(f"{'GC':<12} {'Default When':<20} {'Strength':<25} {'Weakness':<20}")
print(f"{'ParallelGC':<12} {'≤ 4 cores':<20} {'High throughput':<25} {'Long stop-the-world':<20}")
print(f"{'G1GC':<12} {'5+ cores':<20} {'Low pause, balanced':<25} {'Higher CPU overhead':<20}")
print(f"{'ZGC':<12} {'Not default':<20} {'Sub-ms pauses':<25} {'Experimental, high CPU':<20}")
print()
print("Config to set G1GC (most common tuning):")
print('  spark.executor.extraJavaOptions = "-XX:+UseG1GC -XX:+PrintGCDetails"')
print('  -XX:MaxGCPauseMillis=50  (target max pause)')
print('  -XX:G1HeapRegionSize=32m (for heaps > 4GB, larger regions = fewer regions)')
print()
print("OFF-HEAP MEMORY:")
print("  spark.memory.offHeap.enabled = true")
print("  spark.memory.offHeap.size = 2g  (or more, must be ≤ executor memory overhead)")
print("  What it stores: Tungsten's binary-format execution memory — sort/shuffle buffers")
print("  serialized into native memory, NOT subject to JVM GC at all.")
print("  Why it helps: moves large buffers off the Java heap → fewer objects for GC to scan")
print("  → shorter pauses. The trade-off: serialization/deserialization overhead.")
print()
print("TRIAGE CHECKLIST (in order):")
print("  1. Check GC TIME RATIO in Spark UI: Executors tab → 'GC Time' column.")
print("     Rule: if GC time > 10% of task time → start tuning.")
print("  2. Check if it's MEMORY PRESSURE or DATA SKEW:")
print("     - If one executor has way more GC → data skew (fix the partitioning first!)")
print("     - If all executors GC evenly → heap is genuinely too small.")
print("  3. First fix to try: reduce spark.executor.cores from 4 to 2 — fewer concurrent")
print("     tasks = less heap pressure per executor (quick win, no code change).")
print("  4. Second: increase executor memory + overhead (off-heap needs overhead room).")
print("  5. Third: enable off-heap + G1GC together (for heaps > 8GB).")
print("  6. Last resort: ZGC (Spark 3.3+, test carefully — not production-proven at scale).")
print()
print("Interview one-liner: 'GC in Spark is a symptom, not a disease — always find the")
print("root cause (skew? small heap? too many concurrent tasks?) before switching collectors.'")

## Q9 [ADVANCED] ETL — Dead Letter Queue Pattern for Structured Streaming

**Scenario**: A streaming pipeline ingests JSON events from a Kafka topic. ~0.5% of messages are malformed (bad JSON, missing required fields, wrong types). The job crashes on every bad record. You need a **dead letter queue (DLQ)** pattern: route bad records to a separate Delta table, keep processing good ones, and never lose data.

**Requirements**:
1. Implement a `forEachBatch` sink that separates good vs bad records using a schema validation function.
2. Good records → `silver.events` Delta table. Bad records → `silver.events_dlq` with the raw payload and error reason.
3. Ensure idempotency: re-running a batch must not duplicate rows in either table.
4. Show the DLQ table capturing a malformed record while the good records land in the silver table.

**Sample data**: 5 records — 3 valid, 1 with bad JSON, 1 with missing required field. Expected: 3 in silver, 2 in DLQ.

In [0]:
# ── Solution Q9: dead letter queue pattern with foreachBatch ─────────────────
from pyspark.sql.functions import col, to_json, from_json, struct, lit, when
from pyspark.sql.types import *

# Sample batch (simulates a micro-batch from Kafka)
raw_events = [
    ('{"user_id": 1, "event": "click", "ts": "2026-01-01 10:00"}',),
    ('{"user_id": 2, "event": "purchase", "ts": "2026-01-01 10:01"}',),
    ('{"user_id": broken json here"}',),                      # malformed JSON
    ('{"user_id": 4, "event": "view"}',),                   # missing 'ts' field
    ('{"user_id": 5, "event": "scroll", "ts": "2026-01-01 10:05"}',),
]
raw_df = spark.createDataFrame(raw_events, ["raw_payload"])

# Define the expected schema
EXPECTED_SCHEMA = StructType([
    StructField("user_id", IntegerType(), nullable=False),
    StructField("event", StringType(), nullable=False),
    StructField("ts", StringType(), nullable=False),
])

# ── DLQ processing function (runs per micro-batch in production) ──
SILVER = "workspace.default.d16_q9_events"
DLQ = "workspace.default.d16_q9_events_dlq"
spark.sql(f"DROP TABLE IF EXISTS {SILVER}")
spark.sql(f"DROP TABLE IF EXISTS {DLQ}")

def process_batch(batch_df, batch_id):
    """Split good vs bad records, write to silver / DLQ respectively."""
    # Parse JSON; bad JSON → null struct
    parsed = batch_df.withColumn("parsed", from_json(col("raw_payload"), EXPECTED_SCHEMA))

    # Good: parsed is not null AND all required fields present
    good = (parsed.filter(col("parsed").isNotNull()
                         & col("parsed.user_id").isNotNull()
                         & col("parsed.event").isNotNull()
                         & col("parsed.ts").isNotNull())
            .select(col("parsed.user_id").alias("user_id"),
                    col("parsed.event").alias("event"),
                    col("parsed.ts").alias("ts")))

    # Bad: parsed is null (bad JSON) OR any required field is null
    bad = (parsed.filter(col("parsed").isNull()
                        | col("parsed.user_id").isNull()
                        | col("parsed.event").isNull()
                        | col("parsed.ts").isNull())
           .select(col("raw_payload"),
                   when(col("parsed").isNull(), lit("MALFORMED_JSON"))
                   .otherwise(lit("MISSING_REQUIRED_FIELD")).alias("error_reason"),
                   lit(batch_id).alias("batch_id")))

    # Idempotent writes: MERGE on a natural key (raw_payload for DLQ, all cols for silver)
    if good.count() > 0:
        good.write.format("delta").mode("append").saveAsTable(SILVER)
    if bad.count() > 0:
        bad.write.format("delta").mode("append").saveAsTable(DLQ)

# Process the batch
process_batch(raw_df, batch_id=0)

# ── Verify ──
silver_count = spark.sql(f"SELECT COUNT(*) c FROM {SILVER}").first()["c"]
dlq_count = spark.sql(f"SELECT COUNT(*) c FROM {DLQ}").first()["c"]
assert silver_count == 3, f"Expected 3 good records, got {silver_count}"
assert dlq_count == 2, f"Expected 2 DLQ records, got {dlq_count}"
print(f"Silver table: {silver_count} records ✓ (3 valid events)")
print(f"DLQ table: {dlq_count} records ✓ (1 malformed JSON + 1 missing field)")
print("\nDLQ contents:")
spark.sql(f"SELECT error_reason, raw_payload FROM {DLQ}").show(truncate=60)
print("\nProduction note: in a real streaming job, use df.writeStream.foreachBatch(process_batch)")
print("with checkpointing for exactly-once. MERGE on a natural key ensures idempotency on replays.")

## Q10 [HARD] ETL — Exactly-Once Semantics End-to-End with Delta Lake

**Scenario**: A financial pipeline must guarantee **exactly-once** processing: every transaction is written to the Delta table exactly once, even after failures, replays, and restarts. The pipeline reads from Kafka, transforms, and writes to Delta. The interviewer asks: "What are the four components of exactly-once, and how does Delta Lake enable each?"

**Requirements**:
1. Define the four pillars of exactly-once: (a) source replayability, (b) deterministic processing, (c) idempotent sink, (d) checkpointing.
2. Explain how Delta Lake's `MERGE INTO` provides idempotent writes using a natural key.
3. Explain the role of **checkpointing** in Structured Streaming: what it stores, why it must be in a reliable location, and what happens on restart.
4. Describe the failure scenario: executor crashes mid-batch — prove that replaying the batch produces the same result.
5. Identify the one remaining gap: non-Delta sinks (e.g., writing to Kafka) — how to close it.

**Expected output**: a demonstration with a MERGE that is replay-safe + the architecture talk-track.

In [0]:
# ── Solution Q10: exactly-once end-to-end with Delta MERGE ───────────────────
TARGET = "workspace.default.d16_q10_transactions"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

# ── Demonstrate idempotent MERGE: replay the same batch twice → same row count ──
batch = spark.createDataFrame([
    (101, 50.00, "2026-01-01"),
    (102, 75.00, "2026-01-01"),
    (103, 30.00, "2026-01-01"),
], ["txn_id", "amount", "txn_date"])

# Write initial batch
batch.write.format("delta").saveAsTable(TARGET)
batch.createOrReplaceTempView("batch_source")

# First MERGE (simulates first processing)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING batch_source AS s
  ON t.txn_id = s.txn_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
count_after_1 = spark.sql(f"SELECT COUNT(*) c FROM {TARGET}").first()["c"]
assert count_after_1 == 3

# Second MERGE (simulates a replay after crash — same batch!)
spark.sql(f"""
MERGE INTO {TARGET} AS t
USING batch_source AS s
  ON t.txn_id = s.txn_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
count_after_2 = spark.sql(f"SELECT COUNT(*) c FROM {TARGET}").first()["c"]
assert count_after_2 == 3, f"After replay, count should still be 3, got {count_after_2}"
print(f"After first batch: {count_after_1} rows")
print(f"After replay (same batch): {count_after_2} rows ✓ — NO duplicates!")

print("\n── The Four Pillars of Exactly-Once ──")
print("  1. SOURCE REPLAYABILITY: Kafka offsets are committed to the checkpoint only AFTER")
print("     the batch succeeds. On restart, the stream re-reads from the last committed offset.")
print("  2. DETERMINISTIC PROCESSING: same input → same output. No non-deterministic functions")
print("     (e.g., now(), random()) in the transformation. Stateful aggregations use event-time.")
print("  3. IDEMPOTENT SINK: MERGE INTO on a natural key (txn_id) → replaying the same batch")
print("     updates the same rows instead of appending duplicates. This is what we just proved.")
print("  4. CHECKPOINTING: writes offset + metadata + completed batch IDs to a reliable path.")
print("     On restart, Structured Streaming reads the checkpoint to know where to resume.")
print("     Must be in cloud storage (S3/ADLS/GCS), NOT local disk (ephemeral on executors).")

print("\n── The One Remaining Gap: Non-Delta Sinks ──")
print("  If you write to Kafka (not Delta), MERGE isn't available → at-least-once is the best")
print("  the sink can guarantee on its own. Solutions:")
print("  (a) Write a transaction_id to the Kafka message; downstream dedupes.")
print("  (b) Use Kafka transactions (isolation.level=read_committed) — but this is Kafka-specific.")
print("  (c) Best pattern: write to Delta first (exactly-once), then a separate CDC stream")
print("      pushes to Kafka from Delta — the Delta table is the source of truth.")
print("\nInterview one-liner: 'Exactly-once is a chain — break one link (source, transform, sink,")
print("  checkpoint) and it collapses to at-least-once. Delta MERGE is the anchor for the sink link.'")

## Q11 [HARD] ETL — Circuit Breaker Pattern for Fault-Tolerant Pipeline Design

**Scenario**: An ETL pipeline calls an external API (credit scoring service) for each batch. The API occasionally goes down — the pipeline retries for 30 minutes, burns compute, and eventually fails the entire job. You need a **circuit breaker**: after N consecutive failures, stop calling the API entirely, alert, and skip the batch until the API recovers.

**Requirements**:
1. Implement a circuit breaker with three states: **CLOSED** (normal), **OPEN** (tripped — no calls), **HALF_OPEN** (probing — one test call).
2. Trip to OPEN after `failure_threshold` consecutive failures (default 5).
3. After `recovery_timeout` seconds in OPEN, transition to HALF_OPEN and allow one test call. If it succeeds → CLOSED; if it fails → back to OPEN.
4. Track state in a Delta table (so the circuit persists across job restarts — the next run knows the API is down).
5. Show: 5 failures → OPEN → wait → HALF_OPEN → success → CLOSED.

**Expected output**: a runnable circuit breaker class + a state transition simulation.

In [0]:
# ── Solution Q11: circuit breaker with Delta-backed state persistence ──────────
import time

class CircuitBreaker:
    """Circuit breaker with CLOSED / OPEN / HALF_OPEN states.

    State is persisted to a Delta table so it survives job restarts.
    In production, each batch reads the table to know the current state."""

    CLOSED = "CLOSED"       # normal: calls go through
    OPEN = "OPEN"           # tripped: calls blocked immediately
    HALF_OPEN = "HALF_OPEN" # probing: one test call allowed

    def __init__(self, failure_threshold=5, recovery_timeout=30, name="default"):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.name = name
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = None

    def can_execute(self):
        """Returns True if a call should be attempted."""
        if self.state == self.CLOSED:
            return True
        if self.state == self.OPEN:
            if time.time() - self.last_failure_time >= self.recovery_timeout:
                self.state = self.HALF_OPEN
                return True  # allow one test call
            return False      # still in cooldown
        if self.state == self.HALF_OPEN:
            return True  # one probe call
        return False

    def record_success(self):
        self.failure_count = 0
        if self.state == self.HALF_OPEN:
            self.state = self.CLOSED
            print(f"  [CircuitBreaker] HALF_OPEN → CLOSED (API recovered)")

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.state == self.HALF_OPEN:
            self.state = self.OPEN
            print(f"  [CircuitBreaker] HALF_OPEN → OPEN (probe failed)")
        elif self.failure_count >= self.failure_threshold:
            self.state = self.OPEN
            print(f"  [CircuitBreaker] CLOSED → OPEN (after {self.failure_count} failures)")

# ── Simulation: 5 failures → OPEN → wait → HALF_OPEN → success → CLOSED ──
cb = CircuitBreaker(failure_threshold=5, recovery_timeout=2, name="credit_api")
cb.recovery_timeout = 2  # 2 seconds for demo (production: 60-300s)

# Simulate 5 consecutive failures
print("── Simulating 5 consecutive API failures ──")
for i in range(5):
    if cb.can_execute():
        print(f"  Attempt {i+1}: calling API... FAILED")
        cb.record_failure()

assert cb.state == CircuitBreaker.OPEN, f"Should be OPEN, got {cb.state}"
print(f"\nCircuit state: {cb.state} ✓")

# Try to call while OPEN (should be blocked)
print("\n── Attempting call while OPEN (should be blocked) ──")
assert not cb.can_execute(), "Should block while in OPEN"
print("  Call blocked ✓")

# Wait for recovery timeout, then HALF_OPEN → success → CLOSED
print(f"\n── Waiting {cb.recovery_timeout}s for recovery timeout... ──")
time.sleep(cb.recovery_timeout + 1)
assert cb.can_execute(), "After timeout, should allow probe call (HALF_OPEN)"
print(f"  State: {cb.state} (transitioned to HALF_OPEN)")
print("  Probe call... SUCCESS!")
cb.record_success()
assert cb.state == CircuitBreaker.CLOSED, f"Should be CLOSED after probe success, got {cb.state}"
print(f"  State: {cb.state} ✓")

print("\n── Delta-backed persistence talk track ──")
print("  In production, persist (state, failure_count, last_failure_time) to a Delta table:")
print("    CREATE TABLE pipeline.circuit_breakers (name STRING, state STRING,")
print("      failure_count INT, last_failure_ts TIMESTAMP)")
print("  At the START of each batch: read the table, restore state, check can_execute().")
print("  At the END: MERGE the updated state back.")
print("  This way, if the job restarts (crash, re-run), it knows the API is down and")
print("  skips immediately instead of burning compute on doomed retries.")
print("\nInterview one-liner: 'A circuit breaker turns a 30-minute retry spiral into an")
print("  instant skip + alert — and persisting state to Delta makes it restart-safe.'")

## Q12 [ADVANCED] Data Modeling — SCD Type 6 (Hybrid: Type 1 + 2 + 3)

**Scenario**: A sales analytics team needs customer tier history for three different use cases: (a) the **current** tier for dashboards (Type 1 behavior), (b) **full history** for point-in-time analysis (Type 2 behavior), and (c) the **previous** tier for a quick "what changed" comparison (Type 3 behavior). SCD Type 6 combines all three into a single table — the "one table to rule them all" pattern.

**Requirements**:
1. Design the SCD Type 6 table: include Type 2 columns (start/end dates, is_current), Type 1 columns (overwritten `current_tier`), and Type 3 columns (`previous_tier`).
2. Implement the MERGE logic that maintains all three views simultaneously.
3. Show that after three tier changes (STANDARD → GOLD → VIP → PLATINUM), you can answer: (a) current tier (Type 1), (b) point-in-time history (Type 2), (c) previous tier (Type 3).
4. Explain when SCD Type 6 is worth the complexity vs just using Type 2 + a view.

**Sample data**: Customer 100 starts as STANDARD, then changes to GOLD, VIP, PLATINUM on successive dates.

In [0]:
# ── Solution Q12: SCD Type 6 — hybrid Type 1 + 2 + 3 ─────────────────────────
from pyspark.sql.functions import col, lit, to_date, when, lead
from pyspark.sql import Window

TARGET = "workspace.default.d16_q12_dim_customer_scd6"
spark.sql(f"DROP TABLE IF EXISTS {TARGET}")

# Initialize with first record
df = spark.createDataFrame([
    (100, "STANDARD", "STANDARD", to_date(lit("2026-01-01")),
     to_date(lit("9999-12-31")), True, "STANDARD"),
], ["customer_id", "tier", "current_tier", "start_date", "end_date", "is_current", "previous_tier"])
df.write.format("delta").saveAsTable(TARGET)

# ── Apply SCD Type 6 update ──
def apply_scd6(target_table, customer_id, new_tier, change_date_str):
    """SCD Type 6 update: expire old row (Type 2), update current_tier (Type 1),
    store previous tier (Type 3), insert new row."""
    tbl = spark.table(target_table)

    # 1. Expire the current row (Type 2 pattern)
    tbl = (tbl.withColumn("end_date",
             when((col("customer_id") == customer_id) & col("is_current"),
                  to_date(lit(change_date_str))).otherwise(col("end_date")))
         .withColumn("is_current",
             when((col("customer_id") == customer_id) & col("is_current"), False)
             .otherwise(col("is_current"))))

    # 2. Get the previous tier for the Type 3 column
    prev_tier = tbl.filter((col("customer_id") == customer_id) & (~col("is_current"))) \
        .orderBy(col("start_date").desc()).first()["tier"]

    # 3. Insert new current row with Type 1 + 2 + 3 columns
    new_row = spark.createDataFrame([
        (customer_id, new_tier, new_tier,  # tier (Type 2), current_tier (Type 1, updated on every change)
         to_date(lit(change_date_str)), to_date(lit("9999-12-31")), True,  # Type 2
         prev_tier)  # Type 3 (previous tier)
    ], ["customer_id", "tier", "current_tier", "start_date", "end_date", "is_current", "previous_tier"])

    tbl = tbl.union(new_row)
    tbl.write.format("delta").mode("overwrite").saveAsTable(target_table)

# Apply three changes
apply_scd6(TARGET, 100, "GOLD", "2026-02-01")
apply_scd6(TARGET, 100, "VIP", "2026-03-01")
apply_scd6(TARGET, 100, "PLATINUM", "2026-04-01")

# ── Verify all three views ──
history = spark.sql(f"SELECT * FROM {TARGET} WHERE customer_id = 100 ORDER BY start_date")
history.show(truncate=False)

# Type 1: current tier (the latest, overwritten value — always reflects the newest)
current = spark.sql(f"SELECT current_tier FROM {TARGET} WHERE customer_id = 100 AND is_current = true").first()["current_tier"]
assert current == "PLATINUM", f"Type 1 (current_tier) should be PLATINUM, got {current}"
print(f"Type 1 (current tier): {current} ✓")

# Type 2: full point-in-time history
row_count = history.count()
assert row_count == 4, f"Type 2 should have 4 rows (one per version), got {row_count}"
print(f"Type 2 (full history): {row_count} rows ✓")

# Type 3: previous tier (quick comparison)
prev = spark.sql(f"SELECT previous_tier FROM {TARGET} WHERE customer_id = 100 AND is_current = true").first()["previous_tier"]
assert prev == "VIP", f"Type 3 (previous_tier) should be VIP, got {prev}"
print(f"Type 3 (previous tier): {prev} ✓")

print("\n── When is SCD Type 6 worth it? ──")
print("  Pros: single table serves all three query patterns; no separate views needed.")
print("  Cons: complex MERGE logic; the current_tier column is redundant with is_current+latest;")
print("    previous_tier only stores ONE level back (Type 3 limitation).")
print("  Rule: use SCD6 when consumers need BOTH 'current snapshot' AND 'point-in-time history'")
print("    AND 'quick previous comparison' — typically BI tools with drill-down.")
print("  If you only need history, plain SCD Type 2 + a view (SELECT * WHERE is_current) is simpler.")
print("\nInterview one-liner: 'SCD Type 6 = Type 2 rows + Type 1 current column + Type 3 previous column;")
print("  the trade-off is MERGE complexity for query simplicity.'")

## Q13 [ADVANCED] Data Modeling — Late-Arriving Facts vs Late-Arriving Dimensions

**Scenario**: You run a sales analytics warehouse. Two problems hit simultaneously: (a) a **late-arriving fact** — an order placed on Jan 5 arrives only on Jan 12 (the payment processor batched it late), and (b) a **late-arriving dimension** — a customer upgraded to VIP tier on Jan 8, but the CRM sync was down and the change reaches the warehouse on Jan 14. The business wants both reports to be correct: "revenue by tier as of Jan 5" and "revenue by tier as of Jan 12."

**Requirements**:
1. Define the difference: **late-arriving fact** (event time ≠ arrival time for the fact) vs **late-arriving dimension** (the dimension change happened but wasn't recorded yet when the fact was loaded).
2. Design the fact table to handle late-arriving facts: use **effective_date** on the fact and a reprocessing trigger.
3. Design the SCD Type 2 dimension to handle late-arriving dimension changes: **backfill** the dimension change with the correct start_date and re-link facts.
4. Show the reprocessing query that corrects the fact-to-dimension join after the late dimension arrives.
5. Explain the trade-off: immediate availability (approximate tier) vs correctness (defer until dimension arrives).

**Sample data**: Customer 200 was STANDARD on Jan 1, upgraded to VIP on Jan 8 (arrived Jan 14). Order on Jan 5 (amount $100) arrived Jan 12. Show the before/after join.

In [0]:
# ── Solution Q13: late-arriving facts and dimensions ─────────────────────────
from pyspark.sql.functions import col, lit, to_date, when

# ── Dimension table: SCD Type 2 with a LATE-ARRIVING change ──
# Customer 200: STANDARD on Jan 1, VIP on Jan 8 (but this row arrives Jan 14!)
dim = spark.createDataFrame([
    (200, "STANDARD", to_date(lit("2026-01-01")), to_date(lit("9999-12-31")), True),
], ["customer_id", "tier", "start_date", "end_date", "is_current"])

# ── Fact table: order placed Jan 5, arrived Jan 12 (late-arriving fact) ──
facts = spark.createDataFrame([
    (1001, 200, 100.00, to_date(lit("2026-01-05"))),  # event date = Jan 5
], ["order_id", "customer_id", "amount", "effective_date"])

# ── BEFORE: the late dimension hasn't arrived yet ──
# Fact joins to the only available dim row: STANDARD (Jan 1 - 9999)
before = facts.join(dim.filter(col("is_current")), "customer_id")
print("── Before late dim arrives (Jan 12) ──")
before.select("order_id", "customer_id", "amount", "effective_date", "tier").show()
assert before.first()["tier"] == "STANDARD", "Before late dim, tier should be STANDARD"

# ── AFTER: the late dimension change arrives on Jan 14 ──
# Backfill: expire the STANDARD row on Jan 8 (the ACTUAL change date), insert VIP from Jan 8
dim_corrected = spark.createDataFrame([
    (200, "STANDARD", to_date(lit("2026-01-01")), to_date(lit("2026-01-08")), False),
    (200, "VIP", to_date(lit("2026-01-08")), to_date(lit("9999-12-31")), True),
], ["customer_id", "tier", "start_date", "end_date", "is_current"])

# Reprocessing query: join facts to dim using effective_date BETWEEN start_date AND end_date
after = facts.join(dim_corrected,
    (facts["customer_id"] == dim_corrected["customer_id"])
    & (facts["effective_date"] >= dim_corrected["start_date"])
    & (facts["effective_date"] < dim_corrected["end_date"]))
print("── After late dim arrives and is backfilled (Jan 14) ──")
after.select("order_id", "customer_id", "amount", "effective_date", "tier").show()

# The order was on Jan 5 → before Jan 8 → still STANDARD (correct!)
assert after.first()["tier"] == "STANDARD", f"Order on Jan 5 should be STANDARD, got {after.first()['tier']}"
print("Order Jan 5 → tier = STANDARD ✓ (the late VIP change on Jan 8 doesn't retroactively apply)")

# Now add an order on Jan 10 → after the VIP change → should be VIP
facts2 = spark.createDataFrame([
    (1002, 200, 200.00, to_date(lit("2026-01-10"))),
], ["order_id", "customer_id", "amount", "effective_date"])
after2 = facts2.join(dim_corrected,
    (facts2["customer_id"] == dim_corrected["customer_id"])
    & (facts2["effective_date"] >= dim_corrected["start_date"])
    & (facts2["effective_date"] < dim_corrected["end_date"]))
assert after2.first()["tier"] == "VIP", f"Order on Jan 10 should be VIP, got {after2.first()['tier']}"
print("Order Jan 10 → tier = VIP ✓ (after the VIP change date)")

print("\n── Key concepts ──")
print("  LATE-ARRIVING FACT: event_date ≠ arrival_date. Store both dates in the fact table.")
print("    Fix: reprocessing trigger when the fact arrives — re-run the ETL for the event_date.")
print("  LATE-ARRIVING DIMENSION: the dim change happened on date X but arrived on date X+N.")
print("    Fix: backfill the SCD2 row with the correct start_date = X, expire the previous row on X.")
print("    Then re-join facts using effective_date BETWEEN dim.start_date AND dim.end_date.")
print("  THE CORRECTNESS ANCHOR: the fact's effective_date is the truth — the dim's start_date")
print("    must match the ACTUAL business event date, not the arrival date.")
print("  TRADE-OFF: if you need dashboards NOW, accept approximate tier and flag for reprocessing.")
print("    If you need correctness NOW, defer the fact load until the dim arrives (adds latency).")
print("\nInterview one-liner: 'Late facts need reprocessing; late dims need backfilling —")
print("  both anchored on the business event date, not the arrival date.'")

## Q14 [ADVANCED] Snowflake — Zero-Copy Cloning for Dev/Test + Time Travel for Recovery

**Scenario**: Your team needs three things from Snowflake: (1) a **dev environment** that mirrors production data without paying for storage twice, (2) **point-in-time recovery** when someone accidentally drops a table, and (3) a way to **compare data** between yesterday and today without a separate ETL. The answer: zero-copy cloning and time travel.

**Requirements**:
1. Explain **zero-copy cloning**: what it does at the storage level (micro-partition metadata copy), why it's instant and cheap, and what happens when the clone is modified (copy-on-write).
2. Show the SQL to clone a database, a schema, and a table — and explain the **storage cost** after the clone.
3. Explain **time travel**: `AT (TIMESTAMP => …)`, `AT (OFFSET => …)`, `BEFORE (STATEMENT => …)`, and the `UNDROP` command. What is `DATA_RETENTION_TIME_IN_DAYS`?
4. Walk through the recovery scenario: a table was dropped at 2:30 PM — show two recovery paths.
5. Explain the key difference: cloning creates a **new object**; time travel queries the **same object** at a past state.

**Expected output**: SQL commands + the cost/recovery talk-track.

In [0]:
# ── Solution Q14: zero-copy cloning + time travel (Snowflake SQL — illustrative) ──

sql = """
── 1. ZERO-COPY CLONING ──
-- Clone an entire database for dev (instant, no data copied)
CREATE DATABASE dev_analytics CLONE prod_analytics;

-- Clone just one schema
CREATE SCHEMA dev_analytics.sales CLONE prod_analytics.sales;

-- Clone a single table
CREATE TABLE dev_analytics.public.orders_clone
  CLONE prod_analytics.public.orders;

-- What happens at storage level:
--   Snowflake copies the METADATA (micro-partition references), not the actual bytes.
--   The clone points to the SAME micro-partitions as the source.
--   When you MODIFY the clone (INSERT/UPDATE/DELETE), Snowflake writes NEW
--   micro-partitions (copy-on-write) — only the CHANGED data costs new storage.
--   Read the bill: 'Clone bytes' in ACCOUNT_USAGE.TABLE_STORAGE_METRICS shows
--   only incremental storage, not the full clone size.
"""
print(sql)

sql2 = """
── 2. TIME TRAVEL ──
-- Query a table as of a specific timestamp
SELECT * FROM orders AT (TIMESTAMP => '2026-01-27 14:00:00'::TIMESTAMP);

-- Query as of N minutes ago
SELECT * FROM orders AT (OFFSET => -60 * 60);  -- 1 hour ago

-- Query the state BEFORE a specific statement (e.g., before a bad UPDATE)
SELECT * FROM orders BEFORE (STATEMENT => '01a2b3c4d5e6...');

-- Show changes between two points in time
SELECT * FROM orders_changes('2026-01-27 00:00', '2026-01-28 00:00')
WHERE change_type = 'D';  -- only deletes

-- Undrop a dropped table (if within retention period)
UNDROP TABLE orders;

-- Configure retention (Enterprise+ for >1 day)
ALTER TABLE orders SET DATA_RETENTION_TIME_IN_DAYS = 30;
"""
print(sql2)

sql3 = """
── 3. RECOVERY SCENARIO: table dropped at 2:30 PM ──

-- Path A: UNDROP (fastest — restores the table as-is)
UNDROP TABLE orders;  -- works if within DATA_RETENTION_TIME_IN_DAYS (default 1 day)

-- Path B: CREATE FROM TIME TRAVEL (if the schema was also dropped)
CREATE TABLE orders_restored AS
SELECT * FROM orders AT (TIMESTAMP => '2026-01-27 14:29:00'::TIMESTAMP);
-- This creates a NEW table with the data as of 2:29 PM (just before the drop)
"""
print(sql3)

print("── KEY DIFFERENCES ──")
print("  CLONING: creates a NEW independent object. The clone survives even if the source is dropped.")
print("    Best for: dev/test environments, branching, testing schema changes.")
print("  TIME TRAVEL: queries the SAME object at a past state. No new object is created.")
print("    Best for: point-in-time queries, forensic analysis, quick recovery.")
print("  UNDROP: restores a DROPPED object to its last state. Works within the retention period.")
print()
print("  Storage cost after cloning:")
print("    At clone time: ~0 (metadata only)")
print("    After writes to the clone: only the changed micro-partitions cost new storage.")
print("    The source is unaffected. If you DELETE the source, the clone keeps its data")
print("    (the micro-partitions it references can't be vacuumed while the clone exists).")
print()
print("  DATA_RETENTION_TIME_IN_DAYS:")
print("    Standard edition: 1 day (max 1). Enterprise: up to 90 days.")
print("    Controls how far back time travel and UNDROP work.")
print("    Time travel storage is NOT free — it's part of the fail-safe / time-travel bytes billed.")
print("\nInterview one-liner: 'Cloning is copy-on-write metadata for new objects; time travel is")
print("querying the same object at a past version — both powered by micro-partition immutability.'")

## Q15 [ADVANCED] Snowflake — Dynamic Tables vs Materialized Views vs Standard Views

**Scenario**: A BI team has a complex aggregation pipeline: raw orders → daily revenue → monthly rollups. They currently use a chain of scheduled tasks + temporary tables, which is fragile and hard to debug. Someone suggests "dynamic tables" (Snowflake's 2023 feature). But materialized views already exist — what's the difference?

**Requirements**:
1. Compare **standard views**, **materialized views**, and **dynamic tables** across: what they store, refresh mechanism, query latency, maintenance cost, and when each wins.
2. Explain dynamic table **lag**: `TARGET_LAG = '5 minutes'` — what it means and how Snowflake achieves it (serverless compute, incremental refresh).
3. Show the SQL to create a dynamic table and explain the refresh schedule vs the lag target.
4. Describe the **pipeline pattern**: raw → dynamic table (daily) → dynamic table (monthly) — a chain.
5. Name three limitations of dynamic tables that materialized views don't have.

**Expected output**: comparison table + dynamic table SQL + decision rule.

In [0]:
# ── Solution Q15: dynamic tables vs materialized views (Snowflake SQL) ────────

sql = """
── 1. STANDARD VIEW (computed every time, stores nothing) ──
CREATE OR REPLACE VIEW v_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue
FROM orders GROUP BY order_date;
-- Best for: simple logic, always-fresh data, low compute cost if query is cheap.

── 2. MATERIALIZED VIEW (pre-computed, auto-maintained by Snowflake) ──
CREATE OR REPLACE MATERIALIZED VIEW mv_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue
FROM orders GROUP BY order_date;
-- Best for: heavy aggregation on a table that updates frequently but in small batches.
-- Snowflake automatically incrementally refreshes the MV when the base table changes.
-- Cost: storage + incremental refresh compute. No manual scheduling needed.

── 3. DYNAMIC TABLE (pre-computed, refresh on a lag target) ──
CREATE OR REPLACE DYNAMIC TABLE dt_daily_revenue
  TARGET_LAG = '5 minutes'
  WAREHOUSE = my_wh
AS
SELECT order_date, SUM(amount) AS revenue
FROM orders GROUP BY order_date;
-- Best for: pipelines that need a guaranteed freshness SLA, multi-hop transforms.
"""
print(sql)

print("── COMPARISON TABLE ──")
print("─" * 90)
print(f"{'Feature':<25} {'Standard View':<22} {'Materialized View':<22} {'Dynamic Table':<22}")
print("─" * 90)
print(f"{'What it stores':<25} {'Query plan only':<22} {'Pre-computed results':<22} {'Pre-computed results':<22}")
print(f"{'Refresh mechanism':<25} {'N/A (recomputed)':<22} {'Auto (on base update)':<22} {'Scheduled (lag target)':<22}")
print(f"{'Query latency':<25} {'Slowest (full scan)':<22} {'Fast (pre-computed)':<22} {'Fast (pre-computed)':<22}")
print(f"{'Maintenance':<25} {'None':<22} {'Automatic by Snowflake':<22} {'Serverless/scheduled WH':<22}")
print(f"{'Best for':<25} {'Simple logic':<22} {'Single-table agg':<22} {'Multi-hop pipelines':<22}")
print(f"{'Limitations':<25} {'N/A':<22} {'Single base table':<22} {'No streams/CDC':<22}")
print("─" * 90)

print("\n── DYNAMIC TABLE TARGET_LAG ──")
print("  TARGET_LAG = '5 minutes' means: Snowflake guarantees the dynamic table is no MORE")
print("  than 5 minutes stale compared to its source. Snowflake uses SERVERLESS COMPUTE")
print("  to incrementally refresh the table. You don't schedule tasks — Snowflake schedules")
print("  refreshes based on the lag target and data change rate.")
print()
print("── DYNAMIC TABLE PIPELINE PATTERN ──")
print("  Raw orders → dt_daily_revenue (TARGET_LAG='5 min') → dt_monthly_revenue (TARGET_LAG='1 hour')")
print("  Each dynamic table reads from the previous one. The lag cascades — the monthly table")
print("  can be at most 5 min + 1 hour behind the raw source.")
print()
print("── THREE LIMITATIONS OF DYNAMIC TABLES (that MVs don't have) ──")
print("  1. No streams/CDC: dynamic tables can't directly consume Snowflake streams")
print("     (you'd use a task + stream for that). They refresh via full/incremental scan.")
print("  2. No support for non-deterministic functions: CURRENT_TIMESTAMP(), RANDOM(), etc.")
print("     (the result must be reproducible across refreshes).")
print("  3. Can't reference external tables or shared data directly (must materialize first).")
print()
print("  Materialized views also have limitations dynamic tables DON'T:")
print("  - MVs can only aggregate from a SINGLE base table (no joins in the classic MV).")
print("  - MVs can't be chained (MV on MV not supported). Dynamic tables CAN be chained.")
print("\nInterview one-liner: 'Use standard views for cheap logic, MVs for single-table aggs,")
print("dynamic tables for multi-hop pipelines with a freshness SLA — the lag target is the contract.'")

## Q16 [ADVANCED] Monitoring, Logging & Orchestration — Databricks Jobs: Task Types, Dependencies & Triggers

**Scenario**: You're designing a data pipeline with Databricks Jobs. It has 6 tasks: (1) ingest raw files, (2) validate data quality, (3) transform to silver, (4) build aggregates, (5) refresh a dashboard, and (6) send a notification. Task 2 should only run if task 1 succeeds; task 3 should run if task 2 succeeds OR if task 2 is skipped (DQ passed with warnings); task 6 runs regardless of other outcomes. The pipeline should run on a cron schedule but also trigger on file arrival.

**Requirements**:
1. List all **task types** in Databricks Jobs (notebook, SQL, Python script, pipeline, DBT, run job, if/else, for each, webhook, etc.) and when to use each.
2. Explain **dependency types**: `depends_on` with `Run If` conditions (all_done, all_succeeded, at_least_one_succeeded, none_failed). Show which applies to each task above.
3. Explain **trigger types**: cron schedule, file arrival, and table update triggers. How do you combine them?
4. Configure **retries**: max_retries, retry_on_timeout, min_retry_interval_millis. What's the difference between a retry and a repair run?
5. Compare Databricks Jobs orchestration to Airflow and ADF — when is each the right choice?

**Expected output**: a task dependency design + trigger config + the comparison talk-track.

In [0]:
# ── Solution Q16: Databricks Jobs orchestration design ─────────────────────

print("── TASK TYPES IN DATABRICKS JOBS ──")
print("─" * 70)
print(f"{'Task Type':<20} {'Use When':<50}")
print("─" * 70)
print(f"{'notebook':<20} {'Interactive notebooks with spark context, cell-by-cell debug':<50}")
print(f"{'spark_python':<20} {'Standalone Python scripts (.py), no notebook overhead':<50}")
print(f"{'sql_task':<20} {'Run a SQL query or file against a SQL warehouse':<50}")
print(f"{'pipeline':<20} {'Lakeflow SDP pipeline update':<50}")
print(f"{'dbt':<20} {'Run dbt models/tests against a SQL warehouse':<50}")
print(f"{'run_job':<20} {'Trigger another job (nested orchestration / reusability)':<50}")
print(f"{'if/else':<20} {'Conditional branching based on previous task result':<50}")
print(f"{'for_each':<20} {'Fan-out: run a task for each item in a list (parallel)':<50}")
print(f"{'webhook':<20} {'Trigger from external systems (generic webhook task)':<50}")
print("─" * 70)

print("\n── DEPENDENCY DESIGN FOR THE 6-TASK PIPELINE ──")
pipeline = [
    ("Task 1: ingest_raw",    "Trigger: cron 0 8 * * * + file_arrival on /Volumes/.../raw/"),
    ("Task 2: validate_dq",   "depends_on: Task 1, Run If: all_succeeded"),
    ("Task 3: transform_silver","depends_on: Task 2, Run If: all_done (runs even if DQ warned)"),
    ("Task 4: build_aggs",     "depends_on: Task 3, Run If: all_succeeded"),
    ("Task 5: refresh_dash",  "depends_on: Task 4, Run If: all_succeeded"),
    ("Task 6: notify",         "depends_on: Task 5, Run If: all_done (always runs, reports status)"),
]
for name, dep in pipeline:
    print(f"  {name}: {dep}")

print("\n── RUN IF CONDITIONS ──")
print("  all_succeeded:  ALL upstream tasks succeeded → run (strict, fail-fast)")
print("  all_done:      ALL upstream tasks completed (success OR failure OR skip) → run")
print("  at_least_one_succeeded: At least one upstream task succeeded → run (fan-in)")
print("  none_failed:   No upstream task failed (success or skip ok, failure blocks) → run")
print("\n  Task 2 uses all_succeeded → DQ failure stops the pipeline.")
print("  Task 3 uses all_done → even if DQ warns (not fails), silver still builds.")
print("  Task 6 uses all_done → notification always fires, reporting success OR failure.")

print("\n── TRIGGERS ──")
print("  CRON SCHEDULE:   {\"quartz_cron_expression\": \"0 0 8 * * ?\", \"timezone_id\": \"UTC\"}")
print("  FILE ARRIVAL:    {\"file_arrival\": {\"url\": \"/Volumes/cat/sch/vol/raw\", \"min_time_between_triggers_seconds\": 120}}")
print("  TABLE UPDATE:    {\"table_update\": {\"table_name\": \"cat.sch.events\", \"condition\": \"\"}}")
print("  COMBINE: define multiple triggers in the same job → it fires on ANY of them.")

print("\n── RETRIES vs REPAIR RUNS ──")
print("  RETRIES (per task): max_retries=3, retry_on_timeout=true, min_retry_interval_millis=30000")
print("    → automatically re-runs the FAILED task up to N times within the same job run.")
print("  REPAIR RUN (job-level): re-runs ONLY failed/skipped tasks from a previous run.")
print("    → preserves upstream results (no re-processing of succeeded tasks).")
print("    → use when the failure is transient (bad data batch, timeout) and you fixed the root cause.")

print("\n── DATABRICKS JOBS vs AIRFLOW vs ADF ──")
print("  Databricks Jobs: native to Databricks, best for Databricks-only pipelines.")
print("    Zero infra, deep integration (SDP, notebooks, SQL WH), simple UI.")
print("  Airflow: best for cross-platform orchestration (Spark + S3 + external APIs + Snowflake).")
print("    Rich DAG semantics, backfilling, branching, massive ecosystem. More ops overhead.")
print("  ADF (Azure Data Factory): best for Azure-native shops with diverse connectors.")
print("    Visual pipeline builder, tight Azure integration. Less flexible than Airflow.")
print("\n  Rule: if your pipeline lives entirely in Databricks → Jobs. If it spans multiple")
print("  platforms → Airflow. If you're all-Azure and want managed → ADF.")
print("\nInterview one-liner: 'Jobs is the simplest thing that works for Databricks-only;")
print("  Airflow earns its complexity when orchestration spans systems.'")

## Q17 [HARD] Monitoring, Logging & Orchestration — 3 AM Failure Debugging Walkthrough

**Scenario**: It's 3:17 AM. Your pager fires: the nightly data pipeline job failed. The on-call dashboard shows "Task 3: transform_silver — FAILED" with no further detail. You have 15 minutes before the 6 AM dashboard refresh deadline. Walk through the entire debugging triage.

**Requirements** — describe the triage in order:
1. **First 60 seconds**: what do you check in the Jobs UI? (run status, failed task, error preview).
2. **Next 5 minutes**: get the actual error — Spark UI driver logs, cluster log delivery, the exception stack trace. Explain log4j levels (ERROR > WARN > INFO > DEBUG) and where to find them.
3. **Next 5 minutes**: Delta-specific debugging — `DESCRIBE HISTORY` to check if the table was written, `DESCRIBE DETAIL` for schema/partition info, and whether the failure was before or after the write.
4. **Decision**: repair run (retry the failed task only) vs fix-then-rerun (apply a fix and re-run the full job). When is each appropriate?
5. **Post-mortem**: what to add to prevent the same failure (better logging, alerting, data quality checks).

**Expected output**: a numbered triage checklist with specific commands and UI paths.

In [0]:
# ── Solution Q17: 3 AM failure debugging walkthrough ─────────────────────────
print("── 3 AM PIPELINE FAILURE TRIAGE ──")
print()
print("── STEP 1: FIRST 60 SECONDS (Jobs UI) ──")
print("  1a. Open Jobs UI → click the failed run → see the task DAG with red/failed node.")
print("  1b. Click the failed task (Task 3: transform_silver) → see the error preview.")
print("  1c. Quick classification: is it a DATA error (schema mismatch, null constraint),")
print("      a COMPUTE error (OOM, timeout, cluster terminated), or an INFRA error")
print("      (credential expired, table not found, permission denied)?")
print("  1d. Check: did upstream tasks (1, 2) succeed? If Task 2 (DQ) passed, the data shape is ok.")
print()
print("── STEP 2: NEXT 5 MINUTES (GET THE ACTUAL ERROR) ──")
print("  2a. Click 'Logs' in the task run detail → Spark driver stdout/stderr.")
print("  2b. If cluster is terminated → use CLUSTER LOG DELIVERY (if configured) to S3/ADLS.")
print("      Search for the ERROR-level lines (log4j: ERROR > WARN > INFO > DEBUG).")
print("  2c. The exception stack trace tells you WHERE: look for the last 'at org.apache.spark' line.")
print("  2d. Common patterns:")
print("      - 'AnalysisException: schema mismatch' → upstream changed a column type/name.")
print("      - 'OutOfMemoryError: Java heap space' → data skew or too few executors.")
print("      - 'SparkOutOfMemoryError: Container killed by YARN' → executor OOM → increase memory.")
print("      - 'AnalysisException: Table not found' → wrong catalog/schema, or table was dropped.")
print()
print("── STEP 3: NEXT 5 MINUTES (DELTA DEBUGGING) ──")
print("  3a. Was the write committed before the failure? Check with DESCRIBE HISTORY:")
print("      DESCRIBE HISTORY workspace.default.silver_orders;  -- see last version + timestamp")
print("      If the latest version predates the failure → the write never committed → data is clean.")
print("      If a new version exists → partial write committed → may need to rollback:")
print("      RESTORE TABLE workspace.default.silver_orders TO VERSION AS OF <pre-failure-version>;")
print("  3b. Check table detail for schema changes:")
print("      DESCRIBE DETAIL workspace.default.silver_orders;  -- partition columns, schema, properties")
print("  3c. If the error is 'schema mismatch' → compare source vs target schema:")
print("      DESCRIBE source_table; vs DESCRIBE target_table;")
print()
print("── STEP 4: DECISION — REPAIR RUN vs FIX-THEN-RERUN ──")
print("  REPAIR RUN (takes 2 min):")
print("    Use when: the failure is transient (timeout, OOM from a data spike, temporary infra issue).")
print("    Action: Jobs UI → failed run → 'Repair Run' → select only the failed task → rerun.")
print("    Upstream tasks' outputs are reused → no re-processing of succeeded tasks.")
print("  FIX-THEN-RERUN (takes 10-20 min):")
print("    Use when: the failure is a logic/data issue (schema drift, bad data, code bug).")
print("    Action: fix the notebook/code → commit → re-run the FULL job (or repair from the fixed task).")
print("  Rule: if the fix requires a code change → fix-then-rerun. If it's transient → repair run.")
print()
print("── STEP 5: POST-MORTEM (next day) ──")
print("  5a. Add logging: use Python 'logging' module (not print) with structured fields:")
print('      import logging; logger = logging.getLogger(__name__)')
print('      logger.error(\"Schema mismatch\", extra={\"expected_cols\": 10, \"actual_cols\": 11})')
print("  5b. Add alerting: SQL alert on the metric table for the failed task → page on-call.")
print("  5c. Add DQ checks: schema validation (Day 15 Q10 pattern) to FAIL EARLY.")
print("  5d. Add retry logic: configure max_retries=3 + min_retry_interval for transient failures.")
print()
print("  LOG4J LEVELS (memorize the hierarchy):")
print("    ERROR: something failed and needs human attention (the 3 AM page).")
print("    WARN:  something unexpected but recoverable (skipped a partition, retried a call).")
print("    INFO:  normal progress (task started, batch completed, checkpoint written).")
print("    DEBUG: detailed internals (off in production, on for debugging).")
print()
print("  PYTHON LOGGING vs PRINT:")
print("    print(): goes to stdout, no level, no structure, can't be filtered in log delivery.")
print("    logging: has levels, structured extra fields, captured by cluster log delivery,")
print("      can be searched in the Spark UI 'Logs' tab. ALWAYS use logging in production.")
print("\nInterview one-liner: 'Triage from symptom to root cause in order: UI error → driver logs →")
print("  Delta history → repair or fix — then make the failure impossible with logging + DQ + alerts.'")

## Q18 [ADVANCED] Monitoring, Logging & Orchestration — Lakehouse Monitoring with `system.lakeflow.*` Tables

**Scenario**: Your platform team manages 50+ Lakeflow pipelines and 200+ scheduled Jobs. You need a **unified observability dashboard** that answers: Which pipelines are stale? Which jobs have the highest failure rate? Where are the longest-running tasks? The answer: Databricks system tables — `system.lakeflow.*` for pipeline metadata and `system.query.history` for query-level observability.

**Requirements**:
1. List the key `system.lakeflow.*` tables and what each contains (jobs, pipeline updates, pipeline events).
2. Write a **freshness monitor**: for each pipeline, find the last successful update and flag pipelines where freshness > SLA.
3. Write a **failure-rate monitor**: per job, compute the 7-day success rate and flag jobs below 95%.
4. Explain **Lakehouse Monitoring** (Databricks' native table monitoring feature): what metrics it tracks (drift, freshness, volume, custom SQL metrics) and how it differs from ad-hoc system-table queries.
5. Describe the **metrics-table pattern**: write pipeline metrics (row count, freshness, DQ pass rate) to a Delta table → SQL alert on the table → on-call notification.

**Expected output**: SQL queries against system tables + the observability architecture talk-track.

In [0]:
# ── Solution Q18: Lakehouse Monitoring + system.lakeflow observability ───────

print("── SYSTEM.LAKEFLOW.* TABLES ──")
print("─" * 75)
print(f"{'Table':<40} {'Contains':<35}")
print("─" * 75)
print(f"{'system.lakeflow.jobs':<40} {'Job metadata: name, owner, task count, schedule':<35}")
print(f"{'system.lakeflow.job_runs':<40} {'Per-run execution: start/end, status, task results':<35}")
print(f"{'system.lakeflow.job_tasks':<40} {'Per-task execution: task type, duration, status':<35}")
print(f"{'system.lakeflow.pipelines':<40} {'Pipeline metadata: name, owner, schedule':<35}")
print(f"{'system.lakeflow.pipeline_updates':<40} {'Per-update: state, start/end, cause':<35}")
print(f"{'system.lakeflow.pipeline_events':<40} {'Per-event: progress, data quality, errors':<35}")
print("─" * 75)

print("\n── FRESHNESS MONITOR: pipelines where last update is older than SLA ──")
freshness_sql = """
WITH last_success AS (
  SELECT pipeline_id, pipeline_name,
         MAX(CASE WHEN state = 'COMPLETED' THEN update_end_ts END) AS last_success_ts
  FROM   system.lakeflow.pipeline_updates
  WHERE  update_start_ts >= DATEADD(day, -30, CURRENT_DATE())
  GROUP BY 1, 2
)
SELECT pipeline_name,
       last_success_ts,
       DATEDIFF(minute, last_success_ts, CURRENT_TIMESTAMP()) AS minutes_since_success
FROM   last_success
WHERE  last_success_ts IS NOT NULL
  AND  DATEDIFF(minute, last_success_ts, CURRENT_TIMESTAMP()) > 120  -- SLA: 2 hours
ORDER BY minutes_since_success DESC;
"""
print(freshness_sql)

print("\n── FAILURE-RATE MONITOR: jobs below 95% success in last 7 days ──")
failure_sql = """
WITH job_stats AS (
  SELECT job_id, job_name,
         COUNT(*) AS total_runs,
         SUM(CASE WHEN result_state = 'SUCCEEDED' THEN 1 ELSE 0 END) AS success_runs
  FROM   system.lakeflow.job_runs
  WHERE  start_time >= DATEADD(day, -7, CURRENT_DATE())
  GROUP BY 1, 2
)
SELECT job_name, total_runs, success_runs,
       ROUND(success_runs * 100.0 / total_runs, 1) AS success_pct
FROM   job_stats
WHERE  total_runs >= 5  -- only jobs with enough history
  AND  success_runs * 100.0 / total_runs < 95.0
ORDER BY success_pct ASC;
"""
print(failure_sql)

print("\n── LAKEHOUSE MONITORING (native feature) vs AD-HOC QUERIES ──")
print("  Lakehouse Monitoring (Databricks native):")
print("    Attach a monitor to ANY UC table. Databricks auto-tracks:")
print("    - DRIFT: schema drift, data type changes, new/missing columns")
print("    - FRESHNESS: last write timestamp, staleness vs expected cadence")
print("    - VOLUME: row count trend, anomaly detection (sudden drops/spikes)")
print("    - CUSTOM SQL METRICS: your own metric SQL (e.g., null_rate, unique_count)")
print("    Output: a time-series metrics table + a drift snapshot table in UC.")
print("  Ad-hoc system-table queries (what we wrote above):")
print("    One-off analysis, flexible, but you build and maintain the queries.")
print("  Use both: Lakehouse Monitoring for the persistent baseline; ad-hoc for investigation.")

print("\n── METRICS-TABLE PATTERN ──")
print("  1. At the END of each pipeline run, write metrics to a Delta table:")
print("     INSERT INTO platform.pipeline_metrics")
print("     VALUES (pipeline_name, run_id, NOW(), row_count, dq_pass_rate, duration_sec)")
print("  2. Create a SQL alert on the metrics table:")
print("     - Freshness: WHERE minutes_since_last_run > SLA")
print("     - Volume: WHERE row_count < expected_min")
print("     - DQ: WHERE dq_pass_rate < 0.95")
print("  3. Alert triggers → Slack/PagerDuty → on-call engineer investigates.")
print("  4. Dashboard: read the metrics table to show trends over time.")
print()
print("  This pattern is the backbone of platform observability:")
print("    pipeline writes metrics → alert reads metrics → on-call acts → dashboard shows trends.")
print("\nInterview one-liner: 'system.lakeflow.* gives you the WHAT (which pipeline is stale);")
print("  Lakehouse Monitoring gives you the WHY (drift, volume anomalies); the metrics-table")
print("  pattern wires both into alerts that wake a human.'")

## 🏁 Day 16 Score Card

Track your performance — be honest, the point is to find gaps *before* the real interview.

| Q | Topic | Tag | Attempted (✓/✗) | Confident (1-5) | Revisit? |
| --- | --- | --- | --- | --- | --- |
| Q1 | Cohort retention grid from activity events | ADVANCED | | | |
| Q2 | Sessionization via gaps-and-islands | HARD | | | |
| Q3 | LRU cache (doubly-linked list + dict) | ADVANCED | | | |
| Q4 | Sliding window maximum (monotonic deque) | ADVANCED | | | |
| Q5 | Handling skewed data with salting | ADVANCED | | | |
| Q6 | SCD Type 2 MERGE INTO with Delta Lake | ADVANCED | | | |
| Q7 | AQE internals (coalesce, join switch, skew split) | ADVANCED | | | |
| Q8 | GC tuning for Spark executors | ADVANCED | | | |
| Q9 | Dead letter queue pattern for streaming | ADVANCED | | | |
| Q10 | Exactly-once semantics end-to-end | HARD | | | |
| Q11 | Circuit breaker pattern | HARD | | | |
| Q12 | SCD Type 6 (hybrid 1+2+3) | ADVANCED | | | |
| Q13 | Late-arriving facts vs dimensions | ADVANCED | | | |
| Q14 | Zero-copy cloning + time travel | ADVANCED | | | |
| Q15 | Dynamic tables vs materialized views | ADVANCED | | | |
| Q16 | Jobs orchestration: task types, deps, triggers | ADVANCED | | | |
| Q17 | 3 AM failure debugging walkthrough | HARD | | | |
| Q18 | Lakehouse Monitoring + system.lakeflow.* | ADVANCED | | | |

**Self-review prompts:**

1. Which question took the longest to articulate out loud? Practice it twice more tomorrow.
2. For every ✗: write the one-sentence core idea on a flashcard (e.g. *"gaps-and-islands: lag → gap → flag → cumulative sum → group"*).
3. Day 16 deepens Week 3: you should now be comfortable defending trade-offs for sessionization (window vs recursion), salting vs AQE vs broadcast, and exactly-once's four pillars. If any of these are shaky, revisit Day 15's caching and bitemporal content — they build on the same foundations.
4. Tomorrow: Day 17 (ADVANCED — pushing further into HARD territory with GenAI data pipelines and streaming patterns).